# Лабораторная №9 — колоризация изображений (Flickr30k)

Все шесть этапов методички: 1. загрузка фотографий; 2. предобработка, функция `load_images_from_filelist` и train/test; 3. архитектура U-Net; 4. обучение; 5. тестирование; 6. визуализация. Все фрагменты «Допишите код» реализованы. Данные Flickr30k (~9 ГБ) необходимо скачать отдельно, как указано в методичке, либо использовать подготовленную преподавателем выборку 5000 изображений на Google Drive.

## 1. Загрузка данных и подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)
# Папка из методички. Если у вас другое размещение, измените DATA_DIR.
DATA_DIR=Path("/content/drive/MyDrive/Колоризация/Колоризация")
assert DATA_DIR.exists(),f"Загрузите и распакуйте изображения в {DATA_DIR}"
files=sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png"))
print("Найдено изображений:",len(files))
assert files,"Папка не содержит изображений"

## 2. Предобработка — допишите функцию load_images_from_filelist

Вход: градации серого, 128×128×1; целевое изображение: RGB, 128×128×3. Оба массива float32 в [0,1]. Для экономии RAM можно ограничить `max_samples`.

In [ ]:
def load_images_from_filelist(file_list,img_size=(128,128),max_samples=1000):
    paths=list(file_list)
    rng=np.random.default_rng(42)
    if max_samples is not None and len(paths)>max_samples:
        paths=[paths[i] for i in sorted(rng.choice(len(paths),size=max_samples,replace=False))]
    images_gray,images_color=[],[]
    for p in paths:
        try:
            with Image.open(p) as im:
                color=np.asarray(im.convert("RGB").resize(img_size,Image.Resampling.BILINEAR),dtype=np.float32)/255.
            # standard luminance conversion
            gray=(0.299*color[:,:,0]+0.587*color[:,:,1]+0.114*color[:,:,2]).astype("float32")
            images_gray.append(gray[...,None])
            images_color.append(color)
        except (OSError,ValueError) as e:
            print("Пропуск повреждённого файла:",p,e)
    assert images_gray,"Не удалось прочитать изображения"
    return np.stack(images_gray),np.stack(images_color)
x_data,y_data=load_images_from_filelist(files,img_size=(128,128),max_samples=1000)
x_train,x_test,y_train,y_test=train_test_split(x_data,y_data,test_size=0.2,random_state=42)
print("train:",x_train.shape,y_train.shape,"test:",x_test.shape,y_test.shape)

### 2.1. Вывод исходных чёрно-белых и цветных изображений

In [ ]:
fig,axs=plt.subplots(2,4,figsize=(12,6))
for i in range(4):
    axs[0,i].imshow(x_test[i,:,:,0],cmap="gray",vmin=0,vmax=1)
    axs[1,i].imshow(y_test[i])
    for r in range(2):axs[r,i].axis("off")
plt.tight_layout();plt.show()

## 3. Реализация U-Net — допишите код

Сжатие Conv2D/MaxPooling, расширение UpSampling2D, skip connections, выход RGB sigmoid. Компилируем с MAE; цветовые ошибки оцениваются относительно исходного RGB.

In [ ]:
def block(x,f):
    x=layers.Conv2D(f,3,padding="same",activation="relu")(x)
    return layers.Conv2D(f,3,padding="same",activation="relu")(x)
inp=keras.Input((128,128,1))
c1=block(inp,16);p1=layers.MaxPooling2D()(c1)
c2=block(p1,32);p2=layers.MaxPooling2D()(c2)
b=block(p2,64)
u2=layers.UpSampling2D()(b);u2=block(layers.Concatenate()([u2,c2]),32)
u1=layers.UpSampling2D()(u2);u1=block(layers.Concatenate()([u1,c1]),16)
out=layers.Conv2D(3,1,activation="sigmoid")(u1)
model=keras.Model(inp,out)
model.compile(optimizer="adam",loss="mae",metrics=["mse"])
model.summary()

## 4. Обучение — допишите код

In [ ]:
history=model.fit(x_train,y_train,validation_split=0.2,epochs=10,batch_size=16,verbose=1)

## 5. Тестирование — допишите код

In [ ]:
metrics=model.evaluate(x_test,y_test,return_dict=True,verbose=0)
print("Test:",metrics)

## 6. Результат колоризации — допишите код

In [ ]:
pred=model.predict(x_test[:4],verbose=0)
fig,axs=plt.subplots(3,4,figsize=(12,9))
for i in range(4):
    axs[0,i].imshow(x_test[i,:,:,0],cmap="gray")
    axs[1,i].imshow(y_test[i])
    axs[2,i].imshow(np.clip(pred[i],0,1))
    for r in range(3):axs[r,i].axis("off")
plt.tight_layout();plt.show()
plt.plot(history.history["loss"],label="train")
plt.plot(history.history["val_loss"],label="validation")
plt.xlabel("Эпоха");plt.ylabel("MAE");plt.legend();plt.show()

## Вывод

После запуска сравните оригиналы и раскрашенные фотографии, приведите фактическую MAE/MSE и прокомментируйте сложность восстановления цвета из оттенков серого. Метрики не выдуманы.